In [2]:
import pandas as pd
import matplotlib as mtp

pd.set_option('display.max_rows', None)      # toutes les lignes
pd.set_option('display.max_columns', None)   # toutes les colonnes
pd.set_option('display.width', None)         # pas de retour à la ligne forcé
pd.set_option('display.max_colwidth', None)  # contenu des cellules en entier

distance = pd.read_csv("../data/distances.csv")
schedule = pd.read_csv("../data/released_schedule.csv")



FileNotFoundError: [Errno 2] No such file or directory: '../data/distances.csv'

## Liste des Spécifications et des hypothèses

Nous avons relevé ces différentes contraintes dans le projets :

- 7 jours anticipation max
- Poids camion max : 24,5 T
- Volume max : 110,4 m³
- 1 fournisseur livre à une usine
- Lissage (répartition à une semaine dans un second temps)
- Pas de livraison le week end
- Livraison au moins à la date de besoin


## Analyse des données


#### Analyse des données null ou NaN
On vérifie la présence de NaN dans la distance pour les fournisseurs


In [ ]:
distance[distance.isnull().any(axis=1)]


,supplier_id,distance
1,SUP_00002,NaN
11,SUP_00012,NaN
29,SUP_00030,NaN
30,SUP_00031,NaN
46,SUP_00047,NaN
49,SUP_00050,NaN
83,SUP_00084,NaN
114,SUP_00115,NaN
129,SUP_00130,NaN
136,SUP_00137,NaN


On vérifie la présence de NaN ou 0 dans la table schedule

In [ ]:
schedule[schedule.isna().any(axis=1)]


L'analyse révèle la présence de nombreuse valeur NaN et 0, les valeurs 0 sont présentes uniquement pour la colonne schedule_quantity. Les NaN quant à eux sont présents dans les colonnes weight et volumes. Les colonnes ayant 0 dans schedule quantity on forcément leurs valeurs à NaN dans wieght et quantity, le cas inverse n'est pas vrai.

Liste des fournisseurs sans distance mais quand même présent dans le dataset "released_scheduled"

In [ ]:
# Fournisseur ayant des distances manquantes présent dans le planning de livraison
supplier_ids = distance[distance.isnull().any(axis=1)]['supplier_id'].unique()
schedule[schedule['supplier_id'].isin(supplier_ids)]

,date,supplier_id,material,plant,unloading_point,schedule_quantity,weight,volume,transmission_date,version_date
1548,2027-01-01,SUP_00047,MAT_001199,PLT_004,ULP_0027,20.0,140.7,1.20,2026-11-06,2026-11-08
3178,2026-11-17,SUP_00130,MAT_000903,PLT_002,ULP_0008,0.0,0.0,0.42,2026-09-16,2026-11-08
3308,2026-11-20,SUP_00130,MAT_000563,PLT_002,ULP_0008,10.0,142.9,1.20,2026-11-04,2026-11-08
3309,2026-11-26,SUP_00130,MAT_000563,PLT_002,ULP_0008,10.0,142.9,1.20,2026-11-04,2026-11-08
3310,2026-11-27,SUP_00130,MAT_000563,PLT_002,ULP_0008,10.0,142.9,1.20,2026-11-04,2026-11-08
5376,2026-11-17,SUP_00130,MAT_000010,PLT_001,ULP_0028,1.0,63.0,0.42,2026-11-04,2026-11-08


#### Analyse des valeurs abbérantes des datasets


In [ ]:
import matplotlib.pyplot as plt

# --- distances ---
fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(distance["distance"].dropna())
ax.set_title("distances : distance")
ax.set_xticks([])
plt.show()

# --- schedule : une colonne par sous-graphique (les échelles sont très différentes) ---
cols = ["schedule_quantity", "weight", "volume"]
livraisons = schedule[schedule["schedule_quantity"] > 0]  # on enlève les jours sans livraison

fig, axes = plt.subplots(1, len(cols), figsize=(12, 5))
for ax, col in zip(axes, cols):
    ax.boxplot(livraisons[col].dropna())
    ax.set_title(col)
    ax.set_xticks([])
    if col != "volume":
        ax.set_yscale("log")

fig.suptitle("released_schedule (quantité > 0)")
plt.tight_layout()
plt.show()


## Dictionnaire des données

#### Table `distances.csv`
Distance entre chaque fournisseur et le point de consolidation. 156 lignes, une par fournisseur.

| Champ | Description | Type | Contraintes | Exemple |
|---|---|---|---|---|
| supplier_id | Identifiant du fournisseur | Texte | Clé primaire, unique, non nul | SUP_00001 |
| distance | Distance fournisseur → point de consolidation | Décimal | ≥ 0, peut être nul | 47 |

#### Table `released_schedule.csv`
Programme de livraison (release schedule) envoyé aux fournisseurs. 5 728 lignes, une par date × fournisseur × article × usine × quai.

| Champ | Description | Type | Contraintes | Exemple |
|---|---|---|---|---|
| date | Date de livraison demandée | Date | Non nul | 2026-11-11 |
| supplier_id | Fournisseur qui expédie | Texte | Clé étrangère → distances.supplier_id | SUP_00091 |
| material | Référence de la pièce | Texte | Non nul | MAT_000110 |
| plant | Usine de destination | Texte | Non nul | PLT_004 |
| unloading_point | Quai de déchargement dans l'usine | Texte | Non nul | ULP_0050 |
| schedule_quantity | Quantité à livrer | Décimal | ≥ 0 ; 0 = pas de besoin | 5 000 |
| weight | Poids de la ligne | Décimal | ≥ 0, peut être nul | 92,6 |
| volume | Volume de la ligne | Décimal | > 0, peut être nul | 0,4116 |
| transmission_date | Date d'envoi du programme au fournisseur | Date | Non nul, ≤ version_date | 2026-11-06 |
| version_date | Version du programme | Date | Valeur unique | 2026-11-08 |

#### Clés et relations
| Élément | Définition |
|---|---|
| Clé primaire schedule | (date, supplier_id, material, plant, unloading_point), sans doublon |
| Clé de consolidation | fournisseur × usine × date |



In [1]:
# Nombre de lignes contenant des quantités à 0
n = (schedule["schedule_quantity"] == 0).sum()
print(f"{n} lignes à 0 ({n / len(schedule):.1%})")

NameError: name 'schedule' is not defined